# Loomic Extractor — Demo

Extracts the Loom knowledge graph from Loomic-0.2 annotated Markdown (see `Loomic-Spec-v0.2.md`).

Requirements: `pyyaml` (usually present). Optional: `networkx` (GraphML/graph export), `pandas` (dataframe views).
This notebook assumes it lives in the SAIR folder root, next to `loomic.py`.

In [ ]:
# One-time setup: install optional dependencies into this kernel if missing
import importlib.util
missing = [p for p in ("yaml", "networkx", "pandas", "matplotlib", "pyvis")
           if importlib.util.find_spec(p) is None]
if missing:
    pkgs = ["pyyaml" if p == "yaml" else p for p in missing]
    %pip install -q {" ".join(pkgs)}
    print("installed:", ", ".join(pkgs), "— restart the kernel if imports below fail")
else:
    print("all dependencies present")

In [ ]:
from loomic import Loom

BASE = "SAIR/SAIR UCR 2026"
loom = Loom.from_paths([
    f"{BASE}/presentations-loomic",   # Loomic pilot files
    f"{BASE}/people",                 # plain md -> document nodes (resolve refs)
    f"{BASE}/concepts",
])
loom.report()

## Inspect a node and its edges

In [ ]:
n = loom.node("barish-confidence-gap")
print(n["label"], "\n")
print("attrs:  ", n["attrs"])
print("derived:", n["derived"])
print("\nedges:")
for e in loom.edges_of("barish-confidence-gap"):
    print(f"  {e['src']} -{e['kind']}@{e['qualifier']}-> {e['dst']}")

## Unknowledge, frontiers, Pendex

In [ ]:
for u in loom.unknowledge("open"):
    print("?", u["id"], "—", u["label"][:80])
print()
for p in loom.pendex():
    print("~", p["id"], "—", p["attrs"].get("trigger.capability") or p["attrs"].get("trigger.event"))

## Diachronic simulation — April 2027

Imagine Loom itself resolves Barish's discovery-confidence gap. We add the (hypothetical)
resolution document **without touching the 2026 files** and watch the graph re-derive:
the unknowledge resolves, the claim is *outdated* (not superseded — Barish was right at the time),
the future binding fires, and downstream nodes are tainted for review.

In [ ]:
FUTURE = '''---
title: "Loom validation framework v1 (hypothetical)"
date: 2027-04-12
loomic:
  loom: sair-ucr-2026
  asserted: 2027-04-12
---

::: {#loom-validation-v1 .resolution
     resolves=post-five-sigma
     outdates=barish-confidence-gap
     parents="barish-confidence-gap@motivated"}
Hypothetical April 2027 event: a Loom-based discovery-confidence
certification framework is accepted by a major collaboration.
:::
'''
loom.add_text(FUTURE, filename="2027-04-loom-validation.md")
loom.report()

## Time travel — the graph as of June 30, 2026

In [ ]:
view_2026 = loom.as_of("2026-06-30")
print("2026 view:", view_2026.node("barish-confidence-gap")["derived"])
print("2027 view:", loom.node("barish-confidence-gap")["derived"])

## Review queue (taint propagation)

In [ ]:
for t in loom.taints():
    print("!", t["id"])
    for c in t["derived"]["review_causes"]:
        print("    cause:", c)

## Exports

In [ ]:
loom.to_json("loom-export.json")
print("wrote loom-export.json")

try:
    loom.to_graphml("loom-export.graphml")   # open in Gephi / yEd / Cytoscape
    print("wrote loom-export.graphml")
except ImportError:
    print("pip install networkx for GraphML export")

try:
    display(loom.nodes_df().query("~stub").head(20))
except ImportError:
    print("pip install pandas for dataframe views")

## Quick visualization (optional, needs networkx + matplotlib)

In [ ]:
try:
    import networkx as nx
    import matplotlib.pyplot as plt
    g = loom.to_networkx()
    # Semantic core only: drop document nodes and stubs
    keep = [n for n, d in g.nodes(data=True)
            if d.get("type") not in ("document",) and d.get("stub") != "True"]
    sub = g.subgraph(keep)
    colors = {"claim": "#e74c3c", "unknowledge": "#9b59b6", "frontier": "#8e44ad",
              "future_binding": "#f39c12", "observation": "#95a5a6", "resolution": "#27ae60",
              "synthesis": "#3498db", "interpretation": "#2980b9", "hypothesis": "#e67e22"}
    node_colors = [colors.get(sub.nodes[n].get("type"), "#bdc3c7") for n in sub]
    plt.figure(figsize=(14, 10))
    pos = nx.spring_layout(sub, seed=42, k=0.6)
    nx.draw_networkx(sub, pos, node_color=node_colors, node_size=700,
                     font_size=7, edge_color="#cccccc", arrows=True)
    plt.axis("off"); plt.title(f"Loom: {loom.name}"); plt.show()
except ImportError as e:
    print("optional:", e)

## Interactive graph (pyvis)

Writes a self-contained HTML file — draggable, zoomable, searchable. Node shapes: diamond = unknowledge, hexagon = frontier, star = future binding, triangle = resolution. Red borders mark tainted (review-needed) nodes; hover for full attributes.

In [ ]:
loom.to_html("loom-interactive.html")
print("wrote loom-interactive.html — open it in your browser")

# Or embed right here in the notebook:
from IPython.display import IFrame
IFrame("loom-interactive.html", width="100%", height=650)

## Layered derivation DAG

The directed view that spring layout can't give you: ancestry flows downward. Sources/talks at top, transcript segments and claims in the middle, unknowledge and resolutions at the bottom — the epistemic supply chain.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

dag = loom.derivation_dag()
for layer, nodes in enumerate(nx.topological_generations(dag)):
    for n in nodes:
        dag.nodes[n]["layer"] = layer
pos = nx.multipartite_layout(dag, subset_key="layer", align="horizontal")
pos = {n: (x, -y) for n, (x, y) in pos.items()}   # roots at top

colors = [Loom.TYPE_COLORS.get(dag.nodes[n].get("type"), "#bdc3c7") for n in dag]
plt.figure(figsize=(16, 9))
nx.draw_networkx_edges(dag, pos, edge_color="#cccccc", arrows=True,
                       connectionstyle="arc3,rad=0.08")
nx.draw_networkx_nodes(dag, pos, node_color=colors, node_size=600)
nx.draw_networkx_labels(dag, pos, font_size=7, verticalalignment="bottom")
edge_labels = {(u, v): d["kind"] for u, v, d in dag.edges(data=True)
               if d["kind"] in ("resolves", "outdates")}
nx.draw_networkx_edge_labels(dag, pos, edge_labels, font_size=6, font_color="#e74c3c")
plt.axis("off"); plt.title(f"Derivation DAG: {loom.name}"); plt.tight_layout(); plt.show()